In [12]:
# Summary statistics
# Table 4 and 5
from pathlib import Path
import pandas as pd
import numpy as np
from ChildProject.projects import ChildProject
from ChildProject.annotations import AnnotationManager

DATA_PATH =  Path.cwd().parent.parent / 'data'

# Read measures
human_measures = pd.read_csv(DATA_PATH / 'measures' / 'human_measures_chunks.csv').fillna(0)

# Read metadata
children = pd.read_csv(DATA_PATH / 'metadata' / 'children.csv')
recordings = pd.read_csv(DATA_PATH / 'metadata' / 'recordings.csv')
recordings_data = recordings.merge(children, on='child_id')[['group_id', 'recording_filename', 'child_sex']]
human_measures = human_measures.merge(recordings_data, how='left', on='recording_filename')

def compute_CVC(data):
    if 'can_voc_CHI' in data.columns and 'non_can_voc_CHI' in data.columns:
        data['CVC'] = data['can_voc_CHI'] + data['non_can_voc_CHI']
    return data

human_measures = compute_CVC(human_measures)

def group_data(data):
    if 'can_voc_CHI' in data.columns and 'non_can_voc_CHI' in data.columns:
        data['CVC'] = data['can_voc_CHI'] + data['non_can_voc_CHI']
    data = data.groupby('recording_filename').agg({
        '5s_CTC': np.sum,
        'voc_dur_chi': np.sum,
        'voc_dur_och': np.sum,
        'voc_dur_mal': np.sum,
        'voc_dur_fem': np.sum,
        'voc_chi': np.sum,
        'wc_adu': np.sum,
        'CVC': np.sum,
        'group_id': 'first',
        'child_id': 'first'
    }).reset_index()
    return data

df = group_data(human_measures)
duration_cols = ['voc_dur_chi', 'voc_dur_och', 'voc_dur_mal', 'voc_dur_fem']
for col in duration_cols:
    df[col] = df[col] / (1000 * 60)  # Convert from milliseconds to minutes

/tmp/ipykernel_26836/320785613.py:29: FutureWarning: The provided callable <function sum at 0x7357466e2dd0> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  data = data.groupby('recording_filename').agg({
/tmp/ipykernel_26836/320785613.py:29: FutureWarning: The provided callable <function sum at 0x7357466e2dd0> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  data = data.groupby('recording_filename').agg({


In [13]:
human_measures.drop_duplicates('child_id').groupby('group_id')['child_sex'].apply(lambda x: (x == 'f').sum())

group_id
angelman_syndrome     6
autism_sibling        7
down_syndrome         4
fragile_x_syndrome    3
low_risk              4
Name: child_sex, dtype: int64

In [15]:
import pandas as pd
import numpy as np

# Calculate total duration for each recording and get statistics
df['total_duration'] = df[duration_cols].sum(axis=1)

metrics = {
    'Cumulated speech/vocalization duration (mn)': df['total_duration'],
    "Key child's vocalizations (%)": (df['voc_dur_chi'] / df['total_duration']) * 100,
    "Other children's vocalizations (%)": (df['voc_dur_och'] / df['total_duration']) * 100,
    'Adult female speech (%)': (df['voc_dur_fem'] / df['total_duration']) * 100,
    'Adult male speech (%)': (df['voc_dur_mal'] / df['total_duration']) * 100,
    'Conversational Turn Count': df['5s_CTC'],
    'Adult Word Count': df['wc_adu'],
    'Child Vocalization Count': df['CVC']
}

# Calculate mean, min, and max for each metric
summary = pd.DataFrame({
    'Mean': {k: v.mean() for k, v in metrics.items()},
    'Min': {k: v.min() for k, v in metrics.items()},
    'Max': {k: v.max() for k, v in metrics.items()}
})

summary = summary.round(1)
print(summary)

                                              Mean    Min     Max
Cumulated speech/vocalization duration (mn)    6.6    1.3    13.6
Key child's vocalizations (%)                 33.0    4.0    60.8
Other children's vocalizations (%)            13.5    0.0    39.1
Adult female speech (%)                       40.6   10.9    75.4
Adult male speech (%)                         13.0    0.0    72.4
Conversational Turn Count                     93.2    0.0   274.0
Adult Word Count                             827.2  142.0  1991.0
Child Vocalization Count                     142.5    7.0   312.0


In [16]:
# Group by group_id and calculate statistics for each group
group_stats = {}
for group in df['group_id'].unique():
    group_data = df[df['group_id'] == group]
    
    group_metrics = {
        'Cumulated speech/vocalization duration (mn)': group_data['total_duration'],
        "Key child's vocalizations (%)": (group_data['voc_dur_chi'] / group_data['total_duration']) * 100,
        "Other children's vocalizations (%)": (group_data['voc_dur_och'] / group_data['total_duration']) * 100,
        'Adult female speech (%)': (group_data['voc_dur_fem'] / group_data['total_duration']) * 100,
        'Adult male speech (%)': (group_data['voc_dur_mal'] / group_data['total_duration']) * 100,
        'Conversational Turn Count': group_data['5s_CTC'],
        'Adult Word Count': group_data['wc_adu'],
        'Child Vocalization Count': group_data['CVC']
    }
    
    group_summary = pd.DataFrame({
        'Mean': {k: v.mean() for k, v in group_metrics.items()},
        'Min': {k: v.min() for k, v in group_metrics.items()},
        'Max': {k: v.max() for k, v in group_metrics.items()}
    })
    
    group_stats[group] = group_summary

# Print statistics for each group
for group, stats in group_stats.items():
    print(f"\nGroup: {group}")
    print("-" * 50)
    print(stats.round(1))


Group: low_risk
--------------------------------------------------
                                              Mean    Min     Max
Cumulated speech/vocalization duration (mn)    6.6    1.3    12.1
Key child's vocalizations (%)                 40.0    4.0    50.6
Other children's vocalizations (%)            14.8    0.0    39.1
Adult female speech (%)                       37.7   10.9    62.0
Adult male speech (%)                          7.5    0.0    26.3
Conversational Turn Count                     94.0   11.0   201.0
Adult Word Count                             721.8  242.0  1266.0
Child Vocalization Count                     142.9    7.0   264.0

Group: down_syndrome
--------------------------------------------------
                                              Mean    Min     Max
Cumulated speech/vocalization duration (mn)    7.1    4.4     9.6
Key child's vocalizations (%)                 25.5   10.3    42.1
Other children's vocalizations (%)            22.2    0.2    36.6
A

In [17]:
# Summary statistics: 2-min clips
from pathlib import Path
import pandas as pd
import numpy as np
from ChildProject.projects import ChildProject
from ChildProject.annotations import AnnotationManager


# Read measures
human_measures = pd.read_csv(DATA_PATH / 'measures' / 'human_measures_chunks.csv').fillna(0)

# Read metadata
children = pd.read_csv(DATA_PATH / 'metadata' / 'children.csv')
recordings = pd.read_csv(DATA_PATH / 'metadata' / 'recordings.csv')
recordings_data = recordings.merge(children, on='child_id')[['group_id', 'recording_filename', 'child_sex']]
human_measures = human_measures.merge(recordings_data, how='left', on='recording_filename')

def compute_CVC(data):
    if 'can_voc_CHI' in data.columns and 'non_can_voc_CHI' in data.columns:
        data['CVC'] = data['can_voc_CHI'] + data['non_can_voc_CHI']
    return data

df = compute_CVC(human_measures)
duration_cols = ['voc_dur_chi', 'voc_dur_och', 'voc_dur_mal', 'voc_dur_fem']
for col in duration_cols:
    df[col] = df[col] / (1000)  # Convert from milliseconds to seconds
df['total_duration'] = df[duration_cols].sum(axis=1)

In [18]:
group_stats = {}
for group in df['group_id'].unique():
    group_data = df[df['group_id'] == group]
    
    group_metrics = {
        'Cumulated speech/vocalization duration (mn)': group_data['total_duration'],
        "Key child's vocalizations (%)": (group_data['voc_dur_chi'] / group_data['total_duration']) * 100,
        "Other children's vocalizations (%)": (group_data['voc_dur_och'] / group_data['total_duration']) * 100,
        'Adult female speech (%)': (group_data['voc_dur_fem'] / group_data['total_duration']) * 100,
        'Adult male speech (%)': (group_data['voc_dur_mal'] / group_data['total_duration']) * 100,
        'Conversational Turn Count': group_data['5s_CTC'],
        'Adult Word Count': group_data['wc_adu'],
        'Child Vocalization Count': group_data['CVC']
    }
    
    group_summary = pd.DataFrame({
        'Mean': {k: v.mean() for k, v in group_metrics.items()},
        'Min': {k: v.min() for k, v in group_metrics.items()},
        'Max': {k: v.max() for k, v in group_metrics.items()}
    })
    
    group_stats[group] = group_summary

# Print statistics for each group
for group, stats in group_stats.items():
    print(f"\nGroup: {group}")
    print("-" * 50)
    print(' '.join(stats.columns))
    for row in stats.round(1).values:
        print(' '.join(map(str, row)))


Group: angelman_syndrome
--------------------------------------------------
Mean Min Max
25.1 0.0 103.8
34.6 0.0 100.0
13.2 0.0 87.7
42.9 0.0 100.0
9.3 0.0 98.0
6.1 0.0 67.0
57.8 0.0 302.0
7.7 0.0 58.0

Group: fragile_x_syndrome
--------------------------------------------------
Mean Min Max
24.9 0.0 123.3
41.8 0.0 100.0
5.1 0.0 100.0
38.5 0.0 100.0
14.6 0.0 100.0
6.6 0.0 50.0
51.7 0.0 386.0
9.8 0.0 44.0

Group: low_risk
--------------------------------------------------
Mean Min Max
26.3 0.0 120.2
48.3 0.0 100.0
14.4 0.0 62.3
31.2 0.0 100.0
6.1 0.0 100.0
6.3 0.0 49.0
48.1 0.0 403.0
9.5 0.0 52.0

Group: autism_sibling
--------------------------------------------------
Mean Min Max
27.0 0.0 106.3
47.6 0.0 100.0
10.0 0.0 90.5
26.9 0.0 98.3
15.5 0.0 97.2
6.4 0.0 41.0
52.4 0.0 316.0
11.0 0.0 62.0

Group: down_syndrome
--------------------------------------------------
Mean Min Max
28.3 0.0 104.3
32.0 0.0 100.0
18.7 0.0 87.6
35.2 0.0 100.0
14.1 0.0 100.0
5.7 0.0 40.0
65.7 0.0 300.0
9.6 0.0

In [19]:
group_stats = {}
for group in df['group_id'].unique():
    group_data = df[df['group_id'] == group]
    
    group_metrics = {
        'Cumulated speech/vocalization duration (mn)': group_data['total_duration'],
        "Key child's vocalizations (%)": (group_data['voc_dur_chi'] / group_data['total_duration']) * 100,
        "Other children's vocalizations (%)": (group_data['voc_dur_och'] / group_data['total_duration']) * 100,
        'Adult female speech (%)': (group_data['voc_dur_fem'] / group_data['total_duration']) * 100,
        'Adult male speech (%)': (group_data['voc_dur_mal'] / group_data['total_duration']) * 100,
        'Conversational Turn Count': group_data['5s_CTC'],
        'Adult Word Count': group_data['wc_adu'],
        'Child Vocalization Count': group_data['CVC']
    }
    
    group_summary = pd.DataFrame({
        'Mean': {k: v.mean() for k, v in group_metrics.items()},
        'Min': {k: v.min() for k, v in group_metrics.items()},
        'Max': {k: v.max() for k, v in group_metrics.items()}
    })
    
    group_stats[group] = group_summary

# Print statistics for each group
for group, stats in group_stats.items():
    print(f"\nGroup: {group}")
    print("-" * 50)
    print(stats.iloc[1:].round(1).to_string(index=False))


Group: angelman_syndrome
--------------------------------------------------
 Mean  Min   Max
 34.6  0.0 100.0
 13.2  0.0  87.7
 42.9  0.0 100.0
  9.3  0.0  98.0
  6.1  0.0  67.0
 57.8  0.0 302.0
  7.7  0.0  58.0

Group: fragile_x_syndrome
--------------------------------------------------
 Mean  Min   Max
 41.8  0.0 100.0
  5.1  0.0 100.0
 38.5  0.0 100.0
 14.6  0.0 100.0
  6.6  0.0  50.0
 51.7  0.0 386.0
  9.8  0.0  44.0

Group: low_risk
--------------------------------------------------
 Mean  Min   Max
 48.3  0.0 100.0
 14.4  0.0  62.3
 31.2  0.0 100.0
  6.1  0.0 100.0
  6.3  0.0  49.0
 48.1  0.0 403.0
  9.5  0.0  52.0

Group: autism_sibling
--------------------------------------------------
 Mean  Min   Max
 47.6  0.0 100.0
 10.0  0.0  90.5
 26.9  0.0  98.3
 15.5  0.0  97.2
  6.4  0.0  41.0
 52.4  0.0 316.0
 11.0  0.0  62.0

Group: down_syndrome
--------------------------------------------------
 Mean  Min   Max
 32.0  0.0 100.0
 18.7  0.0  87.6
 35.2  0.0 100.0
 14.1  0.0 100.0
 